# 09 · Zero-Padding Diagnosis
Analyze the effect of 4-second zero-padding on raw decibel (dB) levels by comparing padded clips against unpadded clips across UrbanSound8K and ESC-50.

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path
import time
import hashlib

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import numpy as np
import pandas as pd
import librosa
from scipy import stats
from tqdm import tqdm

from src.preprocessing.intensity import raw_db
print('Successfully imported raw_db from src.preprocessing.intensity')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported raw_db from src.preprocessing.intensity


In [2]:
# Cell 2 – load labels.csv and audio_rawdb.npy (db_pad), verify sha256
labels_path = REPO_ROOT / 'data/processed/labels.csv'
labels = pd.read_csv(labels_path)

rawdb_path = REPO_ROOT / 'data/processed/audio_rawdb.npy'
db_pad = np.load(rawdb_path)

h = hashlib.sha256()
with open(rawdb_path, 'rb') as f:
    while chunk := f.read(65536):
        h.update(chunk)
rawdb_sha = h.hexdigest()

print(f'labels.csv rows     : {len(labels):,}')
print(f'audio_rawdb.npy size: {rawdb_path.stat().st_size:,} bytes')
print(f'audio_rawdb.npy sha : {rawdb_sha}')
print(f'db_pad shape        : {db_pad.shape}')

assert db_pad.shape == (6558,), f'Expected db_pad.shape == (6558,), got {db_pad.shape}'
assert len(labels) == 6558, f'Expected len(labels) == 6558, got {len(labels)}'
print('Cell 2 asserts passed ✓')

labels.csv rows     : 6,558
audio_rawdb.npy size: 52,592 bytes
audio_rawdb.npy sha : 813337332360ac1587df32bea169807ab9d5380836b9d313285aad7e8740e27a
db_pad shape        : (6558,)
Cell 2 asserts passed ✓


In [3]:
# Cell 3 – unpadded audio loading, n_valid and db_valid calculation, save to audio_rawdb_valid.npy
n_clips = len(labels)
n_valid_list = []
db_valid_list = []
failures = []

t0 = time.perf_counter()
for idx, row in tqdm(labels.iterrows(), total=n_clips, desc='Loading unpadded clips', mininterval=5.0):
    rel_path = row['path']
    abs_path = REPO_ROOT / rel_path
    try:
        y_unpadded, _ = librosa.load(abs_path, sr=22050, mono=True)
        n_v = len(y_unpadded)
        db_v = raw_db(y_unpadded)
        n_valid_list.append(n_v)
        db_valid_list.append(db_v)
    except Exception as e:
        failures.append((idx, rel_path, str(e)))

elapsed = time.perf_counter() - t0
print(f'Total computation time: {elapsed:.2f} s ({elapsed/60:.2f} min)')
print(f'Number of failures    : {len(failures)}')

valid_data = np.zeros((n_clips, 2), dtype=np.float64)
valid_data[:, 0] = n_valid_list
valid_data[:, 1] = db_valid_list

out_valid_path = REPO_ROOT / 'data/processed/audio_rawdb_valid.npy'
np.save(out_valid_path, valid_data)
print(f'Saved audio_rawdb_valid.npy to: {out_valid_path}')
print(f'valid_data shape: {valid_data.shape}, dtype: {valid_data.dtype}')

Loading unpadded clips:   0%|          | 0/6558 [00:00<?, ?it/s]

Loading unpadded clips:   2%|▏         | 118/6558 [00:05<04:33, 23.57it/s]

Loading unpadded clips:  11%|█         | 708/6558 [00:10<01:13, 79.09it/s]

Loading unpadded clips:  22%|██▏       | 1443/6558 [00:15<00:46, 110.04it/s]

Loading unpadded clips:  32%|███▏      | 2114/6558 [00:20<00:37, 119.45it/s]

Loading unpadded clips:  41%|████▏     | 2712/6558 [00:25<00:32, 116.81it/s]

... (3 outputs left out)

Loading unpadded clips:  81%|████████  | 5315/6558 [00:46<00:09, 125.43it/s]

Loading unpadded clips:  94%|█████████▍| 6152/6558 [00:51<00:02, 138.06it/s]

Loading unpadded clips: 100%|██████████| 6558/6558 [00:52<00:00, 124.50it/s]

Total computation time: 52.70 s (0.88 min)
Number of failures    : 0
Saved audio_rawdb_valid.npy to: C:\Users\LENOVO\Desktop\CityPulse\data\processed\audio_rawdb_valid.npy
valid_data shape: (6558, 2), dtype: float64


In [4]:
# Cell 4 – asserts on the mathematical relationship
reloaded_valid = np.load(REPO_ROOT / 'data/processed/audio_rawdb_valid.npy')

assert len(failures) == 0, f'Encountered {len(failures)} failures during unpadded audio processing'
assert reloaded_valid.shape == (6558, 2), f'Expected shape (6558, 2), got {reloaded_valid.shape}'
assert reloaded_valid.dtype == np.float64, f'Expected dtype float64, got {reloaded_valid.dtype}'
assert not np.isnan(reloaded_valid).any(), 'NaN values found in valid_data'
assert not np.isinf(reloaded_valid).any(), 'Inf values found in valid_data'

n_valid = reloaded_valid[:, 0]
db_valid = reloaded_valid[:, 1]

# Condition: rows with n_valid <= 88200
mask_short = (n_valid <= 88200)

# Assert mathematical formula: db_pad == db_valid + 10 * log10(n_valid / 88200)
pred_db_pad = db_valid[mask_short] + 10.0 * np.log10(n_valid[mask_short] / 88200.0)
assert np.allclose(db_pad[mask_short], pred_db_pad, atol=1e-3), 'Math formula assertion failed'

# Assert db_valid >= db_pad - 1e-6
assert np.all(db_valid[mask_short] >= db_pad[mask_short] - 1e-6), 'db_valid is smaller than db_pad for clips with n_valid <= 88200'

# Print max absolute error of the first formula
max_abs_err = float(np.max(np.abs(db_pad[mask_short] - pred_db_pad)))
print(f'Max absolute error of db_pad vs (db_valid + 10*log10(n_valid/88200)): {max_abs_err:.10e} dB')
print(f'Rows checked (n_valid <= 88200): {int(np.sum(mask_short)):,} / 6,558')
print('Cell 4 asserts passed ✓')

Max absolute error of db_pad vs (db_valid + 10*log10(n_valid/88200)): 7.3919438179e-06 dB
Rows checked (n_valid <= 88200): 6,357 / 6,558
Cell 4 asserts passed ✓


In [5]:
# Cell 5 – duration summary and Spearman correlations per original_class
df = labels.copy()
df['n_valid'] = n_valid
df['db_pad'] = db_pad
df['db_valid'] = db_valid
df['duration_s'] = df['n_valid'] / 22050.0

for src in ['us8k', 'esc50']:
    src_df = df[df['source'] == src]
    rows = []
    for orig_cls, grp in src_df.groupby('original_class'):
        n_clips = len(grp)
        frac_short = float((grp['n_valid'] < 88200).mean())
        med_dur = float(grp['duration_s'].median())
        corr_pad = float(stats.spearmanr(grp['n_valid'], grp['db_pad']).statistic)
        corr_valid = float(stats.spearmanr(grp['n_valid'], grp['db_valid']).statistic)
        
        rows.append({
            'original_class': orig_cls,
            'count': n_clips,
            'frac_short': frac_short,
            'median_duration_s': med_dur,
            'spearman_pad': corr_pad,
            'spearman_valid': corr_valid
        })
    
    summary_table = pd.DataFrame(rows).set_index('original_class')
    print(f'=== Duration & Correlation Summary: {src.upper()} ===')
    print(summary_table.round(4).to_string())
    print()

=== Duration & Correlation Summary: US8K ===
                 count  frac_short  median_duration_s  spearman_pad  spearman_valid
original_class                                                                     
air_conditioner   1000      0.0030               4.00        0.0146         -0.0037
car_horn           429      0.5268               3.49        0.5943         -0.0143
drilling          1000      0.1950               4.00        0.3959          0.1909
engine_idling     1000      0.0390               4.00        0.0728          0.0191
jackhammer        1000      0.1970               4.00        0.2742          0.0248
siren              929      0.0344               4.00        0.0316         -0.0646
street_music      1000      0.0000               4.00           NaN             NaN

=== Duration & Correlation Summary: ESC50 ===
                count  frac_short  median_duration_s  spearman_pad  spearman_valid
original_class                                                       

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10468\3384379600.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr_pad = float(stats.spearmanr(grp['n_valid'], grp['db_pad']).statistic)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10468\3384379600.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr_valid = float(stats.spearmanr(grp['n_valid'], grp['db_valid']).statistic)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10468\3384379600.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr_pad = float(stats.spearmanr(grp['n_valid'], grp['db_pad']).statistic)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10468\3384379600.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr_valid = float(stats.spearmanr(grp['n_valid'], grp['db_valid']).statistic)
C:\Users\LENOVO\AppData\Local\Te

In [6]:
# Cell 6 – US8K per-original_class comparison (db_pad vs db_valid)
us8k_df = df[df['source'] == 'us8k']

table_pad = us8k_df.groupby('original_class')['db_pad'].agg(['count', 'median', 'mean', 'min', 'max'])
table_valid = us8k_df.groupby('original_class')['db_valid'].agg(['count', 'median', 'mean', 'min', 'max'])

print('=== US8K: db_pad (4 s zero-padded) per original_class ===')
print(table_pad.round(4).to_string())

print('\n=== US8K: db_valid (unpadded) per original_class ===')
print(table_valid.round(4).to_string())

print('\n=== US8K: Classes sorted by median dB ===')
sorted_pad = table_pad['median'].sort_values()
print('db_pad (lowest to highest):')
for rank, (cls, val) in enumerate(sorted_pad.items(), 1):
    print(f'  {rank:2d}. {cls:20s}: {val:.4f} dB')

sorted_valid = table_valid['median'].sort_values()
print('\ndb_valid (lowest to highest):')
for rank, (cls, val) in enumerate(sorted_valid.items(), 1):
    print(f'  {rank:2d}. {cls:20s}: {val:.4f} dB')

gap_pad = table_pad.loc['jackhammer', 'median'] - table_pad.loc['air_conditioner', 'median']
gap_valid = table_valid.loc['jackhammer', 'median'] - table_valid.loc['air_conditioner', 'median']

print(f'\nGap median(jackhammer) - median(air_conditioner) [db_pad]  : {gap_pad:.4f} dB')
print(f'Gap median(jackhammer) - median(air_conditioner) [db_valid]: {gap_valid:.4f} dB')

=== US8K: db_pad (4 s zero-padded) per original_class ===
                 count   median     mean      min      max
original_class                                            
air_conditioner   1000 -25.5242 -27.6014 -53.8122 -12.3179
car_horn           429 -25.5919 -26.8154 -61.7273  -8.8733
drilling          1000 -23.5397 -24.7883 -49.4883  -4.6928
engine_idling     1000 -25.3434 -24.3650 -49.2806  -6.8691
jackhammer        1000 -23.0456 -24.3519 -55.7593 -14.8157
siren              929 -27.2583 -29.1609 -74.8055  -9.7640
street_music      1000 -24.2014 -25.0848 -55.6648  -6.7075

=== US8K: db_valid (unpadded) per original_class ===
                 count   median     mean      min      max
original_class                                            
air_conditioner   1000 -25.5103 -27.5942 -53.8122 -12.3179
car_horn           429 -22.0452 -22.8843 -54.7404  -3.3448
drilling          1000 -23.1697 -23.9146 -48.4774  -4.6928
engine_idling     1000 -25.2836 -24.2577 -49.2806  -6.8691
jac

In [7]:
# Cell 7 – ESC-50 per-original_class comparison (db_pad vs db_valid)
esc50_df = df[df['source'] == 'esc50']

table_pad_esc = esc50_df.groupby('original_class')['db_pad'].agg(['count', 'median', 'mean', 'min', 'max'])
table_valid_esc = esc50_df.groupby('original_class')['db_valid'].agg(['count', 'median', 'mean', 'min', 'max'])

print('=== ESC-50: db_pad (4 s zero-padded/trimmed) per original_class ===')
print(table_pad_esc.round(4).to_string())

print('\n=== ESC-50: db_valid (unpadded 5 s clips) per original_class ===')
print(table_valid_esc.round(4).to_string())

=== ESC-50: db_pad (4 s zero-padded/trimmed) per original_class ===
                count   median     mean      min      max
original_class                                           
car_horn           40 -21.2272 -20.7680 -37.0320  -5.9405
chainsaw           40 -16.0753 -16.7864 -31.2247  -9.7116
engine             40 -20.6004 -20.3077 -34.5669  -6.7094
hand_saw           40 -21.3687 -22.5870 -40.4476 -12.8607
siren              40 -12.3250 -13.3392 -27.8567  -4.1796

=== ESC-50: db_valid (unpadded 5 s clips) per original_class ===
                count   median     mean      min      max
original_class                                           
car_horn           40 -20.9645 -21.2865 -36.6626  -5.9231
chainsaw           40 -16.1840 -16.6303 -31.3855 -10.6697
engine             40 -20.0966 -20.1856 -34.5914  -6.7129
hand_saw           40 -21.2610 -22.5580 -40.6374 -12.9828
siren              40 -12.9386 -13.3861 -28.0844  -4.0661
